# 03 - Modelos predictivos, búsqueda de hiperparámetros y explicabilidad SHAP

En este notebook se entrenan varios modelos de regresión para estimar la concentración de PM2.5, se comparan métricas y se analiza el comportamiento de las variables con SHAP.

In [ ]:
# %pip install scikit-learn matplotlib seaborn pandas numpy shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/who_ambient_air_quality_database_version_2024_(v6.1).xlsx")

raw_df = pd.read_excel(DATA_PATH, sheet_name="Update 2024 (V6.1)")
for col in ["year", "population", "latitude", "longitude", "pm10_concentration", "pm25_concentration", "no2_concentration", "pm10_tempcov", "pm25_tempcov", "no2_tempcov", "who_ms"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

clean_df = raw_df.dropna(subset=["pm25_concentration"]).copy()
clean_df["type_of_stations"] = clean_df["type_of_stations"].fillna("Unknown")
clean_df["country_name"] = clean_df["country_name"].fillna("Unknown")
clean_df["who_region"] = clean_df["who_region"].fillna("Unknown")

features = [
    "year", "population", "latitude", "longitude",
    "pm10_concentration", "no2_concentration",
    "pm10_tempcov", "no2_tempcov",
    "who_region", "country_name", "type_of_stations"
]
target = "pm25_concentration"

X = clean_df[features]
y = clean_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = ["year", "population", "latitude", "longitude", "pm10_concentration", "no2_concentration", "pm10_tempcov", "no2_tempcov"]
categorical_features = ["who_region", "country_name", "type_of_stations"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "LinearRegression": Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())]),
    "RandomForest": Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor(random_state=42, n_estimators=200, n_jobs=-1))]),
    "ExtraTrees": Pipeline([("preprocessor", preprocessor), ("model", ExtraTreesRegressor(random_state=42, n_estimators=200, n_jobs=-1))]),
    "GradientBoosting": Pipeline([("preprocessor", preprocessor), ("model", GradientBoostingRegressor(random_state=42))])
}

results = []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    results.append({
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": np.sqrt(mean_squared_error(y_test, pred)),
        "r2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("r2", ascending=False)
results_df

In [ ]:
# Visualización comparativa
plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x="model", y="r2")
plt.title("Comparación de R² por modelo")
plt.ylabel("R²")
plt.xlabel("Modelo")
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x="model", y="rmse")
plt.title("Comparación de RMSE por modelo")
plt.ylabel("RMSE")
plt.xlabel("Modelo")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Búsqueda de hiperparámetros para el mejor modelo (RandomForest)
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=1))
])

param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2", 0.8]
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)
print("Mejores hiperparámetros:")
print(random_search.best_params_)

best_model = random_search.best_estimator_

pred = best_model.predict(X_test)
print(f"RMSE final: {np.sqrt(mean_squared_error(y_test, pred)):.3f}")
print(f"R2 final: {r2_score(y_test, pred):.3f}")

In [ ]:
# Guardar el mejor modelo
joblib.dump(best_model, "../model/best_air_quality_model.joblib")
print("Modelo guardado en ../model/best_air_quality_model.joblib")

In [ ]:
# SHAP para el mejor modelo
# El preprocesador se aplica primero y luego se obtiene el resultado del modelo
X_test_processed = best_model.named_steps["preprocessor"].transform(X_test)
feature_names = []
feature_names.extend(numeric_features)
feature_names.extend(best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features))

explainer = shap.TreeExplainer(best_model.named_steps["model"])
shap_values = explainer.shap_values(X_test_processed)

# Para modelos con salida vectorial, tomamos la primera salida
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names, show=False)
plt.title("SHAP summary plot")
plt.show()

In [ ]:
# Gráfico de importancia media absoluta de SHAP
shap_df = pd.DataFrame(shap_values, columns=feature_names)
importance = shap_df.abs().mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
importance.plot(kind="bar")
plt.title("Importancia media absoluta de SHAP")
plt.ylabel("Importancia media")
plt.show()

In [ ]:
# Gráfico de dependencia para la variable más importante
most_important = importance.index[0]
if most_important in feature_names:
    shap.dependence_plot(
        most_important,
        shap_values,
        X_test_processed,
        feature_names=feature_names,
        show=False
    )
    plt.title(f"Dependencia SHAP para {most_important}")
    plt.show()